# ETL Transform: Stocks

This notebook runs the **stocks ETL pipeline**: ingest from Postgres (with warmup window) → transform (returns, volatility, technical indicators) → save to `historical_processed` → publish to S3.

**S3 upload modes (set in config cell below):**
- **Per day**: one CSV per book per day at `stocks/transformed/crypto/book={book}/year=.../month=.../day=.../format=csv/YYYYMMDD-{book}.csv`
- **Batch (run/week/month/year)**: one CSV per run (all books) or per (book, partition) at `stocks/transformed/crypto/book={book}/year=.../week=...` or `month=...` or `year=.../format=csv/...`

Set `AWS_DEFAULT_STOCKS_BUCKET` in `.env` for S3 uploads.

In [1]:
import sys
from pathlib import Path

# Resolve project root: run from repo root or notebooks/etl/
_cwd = Path(".").resolve()
project_root = _cwd if (_cwd / "src").is_dir() else (_cwd.parent.parent if _cwd.name == "etl" else _cwd)
src_path = project_root / "src"
if src_path.is_dir():
    sys.path.insert(0, str(project_root))
    sys.path.insert(0, str(src_path))
else:
    raise FileNotFoundError(f"Expected src at {src_path}. Run from repo root or notebooks/etl/.")

import pandas as pd

In [2]:
# Config: date range, books, and S3 upload options
SINCE = "2026-08-30"
UNTIL = "2026-09-20"
BOOKS = ["btc-usd"]  # None = all books; or e.g. ["btc-usd", "eth-usd"]
WARMUP_DAYS = 252
# S3: per-day (one file per book per day) and/or batch (one file per book per week/month/year)
UPLOAD_S3 = True
UPLOAD_S3_BATCH = ["week"]  # e.g. ["run", "week", "month", "year"] or None

In [3]:
# Run stocks ETL: transform (with warmup for indicators), save to Postgres, publish to S3.

import logging
# Show pipeline progress in the notebook (ingest, transform, save, S3 upload)
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%H:%M:%S", force=True)
for _name in ("pipelines.etl_transform", "pipelines.etl_cli", "transform.stocks.stock_transformers"):
    logging.getLogger(_name).setLevel(logging.INFO)

from config.settings import get_settings
from pipelines.etl_transform import run_stocks_etl

_settings = get_settings()
_stocks_bucket = _settings.aws.stocks_bucket
if (UPLOAD_S3 or UPLOAD_S3_BATCH) and not _stocks_bucket:
    print(
        "S3 upload skipped: set AWS_DEFAULT_STOCKS_BUCKET in .env, "
        "then restart Jupyter."
    )

transformed_df = run_stocks_etl(
    since=SINCE,
    until=UNTIL,
    books=BOOKS,
    warmup_days=WARMUP_DAYS,
    stocks_bucket=None,  # uses AWS_DEFAULT_STOCKS_BUCKET from .env
    save_to_postgres=False,
    upload_s3=UPLOAD_S3,
    upload_s3_batch=UPLOAD_S3_BATCH,
)

print(f"Transformed {len(transformed_df)} stock records")

15:26:37 - INFO - Ingesting stocks (warmup 2025-12-21 to 2026-09-20)...
15:26:37 - INFO - ============================================================
15:26:37 - INFO - INGEST STOCKS
15:26:37 - INFO - ============================================================
15:26:37 - INFO - Fetching stocks from database...
15:26:37 - INFO -   Books: ['btc-usd']
15:26:37 - INFO - Retrieved 8 records
15:26:37 - INFO - Filtered since 2025-12-21: 8 records
15:26:37 - INFO - Filtered until 2026-09-20: 8 records
15:26:37 - INFO - Ingestion complete: 8 records
15:26:37 - INFO - Transforming (returns, volatility, technical indicators)...
15:26:37 - INFO - Stock transformation pipeline initialized
15:26:37 - INFO - Transforming 8 stock records...
15:26:37 - INFO - Stock transformation complete: 8 records
15:26:37 - INFO - Transformed 8 stock records (2026-08-30 to 2026-09-20)
15:26:37 - INFO - Uploading 8 book/day files to s3://test-financial-stocks-bucket/...


Connection to the database successful!
Table name set to: historical
Connection closed.


15:26:37 - INFO - Uploaded 8 group files to s3://test-financial-stocks-bucket/
15:26:37 - INFO - Uploaded stocks batch week (book=btc-usd, partition 2026-09-13) to s3://test-financial-stocks-bucket/stocks/transformed/crypto/book=btc-usd/year=2026/week=36/format=csv/y2026_w36-btc-usd.csv
15:26:37 - INFO - Uploaded stocks batch week (book=btc-usd, partition 2026-09-14) to s3://test-financial-stocks-bucket/stocks/transformed/crypto/book=btc-usd/year=2026/week=37/format=csv/y2026_w37-btc-usd.csv


Transformed 8 stock records


In [4]:
# Inspect transformed output
if not transformed_df.empty:
    display(transformed_df.head())
    print(transformed_df.columns.tolist())

,ref,book,date,open,high,low,close,adj_close,volume,created_at,...,sma_200,ema_12,ema_26,rsi_14,macd,macd_signal,macd_histogram,bb_upper,bb_middle,bb_lower
0,https://finance.yahoo.com,btc-usd,2026-09-13,77270.18,77415.66,76498.38,76838.16,76838.16,13607872549,2026-09-20 14:48:41.009026,...,NaN,76838.160000,76838.160000,NaN,0.000000,0.000000,0.000000,NaN,NaN,NaN
1,https://finance.yahoo.com,btc-usd,2026-09-14,76830.28,79568.95,76367.38,78163.38,78163.38,29909358382,2026-09-20 14:48:40.869336,...,NaN,77042.040000,76936.324444,NaN,105.715556,21.143111,84.572444,NaN,NaN,NaN
2,https://finance.yahoo.com,btc-usd,2026-09-15,78162.48,78242.55,74944.59,75612.51,75612.51,39694675220,2026-09-20 14:48:40.731229,...,NaN,76822.112308,76838.264115,NaN,-16.151808,13.684127,-29.835935,NaN,NaN,NaN
3,https://finance.yahoo.com,btc-usd,2026-09-16,75611.80,76474.08,74995.52,76150.32,76150.32,30053315577,2026-09-20 14:48:40.659042,...,NaN,76718.759645,76787.305292,NaN,-68.545647,-2.761827,-65.783819,NaN,NaN,NaN
4,https://finance.yahoo.com,btc-usd,2026-09-17,76152.05,77079.16,75945.55,76403.77,76403.77,21917458567,2026-09-20 14:48:40.408803,...,NaN,76670.299700,76758.895270,NaN,-88.595571,-19.928576,-68.666995,NaN,NaN,NaN


['ref', 'book', 'date', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'created_at', 'simple_return', 'log_return', 'volatility_20d', 'volatility_60d', 'volatility_parkinson', 'volatility_gk', 'sma_20', 'sma_50', 'sma_200', 'ema_12', 'ema_26', 'rsi_14', 'macd', 'macd_signal', 'macd_histogram', 'bb_upper', 'bb_middle', 'bb_lower']


## Optional: Specific books and date range

In [5]:
# transformed_df = run_stocks_etl(
#     since="2026-01-01",
#     until="2026-01-28",
#     books=["btc-usd", "eth-usd"],
#     warmup_days=252,
#     save_to_postgres=True,
#     upload_s3=True,
# )